In [ ]:
import sys
import os
import copy
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader

sys.path.append('..')

from src.data.dataset import CustomDataset, CashedCustomDataset
from src.data.degredation import get_transforms
from src.models.mlp import Mlp
from src.models.resnet import resnet18
from src.training.train import train, test

from custom.figure import mm, color
import matplotlib.pyplot as plt

from custom.figure import mm, color
import matplotlib.pyplot as plt


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Determine the project root and dataset directory
dataset = "cifar10c"
dataset_dir = '../dataset'
percent = "0.5pct"

# Load the training and validation datasets using the pre-prepared transform
train_dataset = CashedCustomDataset(dataset, dataset_dir, split="train", percent=percent,
                                transform=get_transforms(dataset, blur=0, color=1, test=False, exclude_to_tensor=True))
train_degradation_dataset = CashedCustomDataset(dataset, dataset_dir, split="train", percent=percent,
                                            transform=get_transforms(dataset, blur=7, color=0, test=False, exclude_to_tensor=True))
test_align_dataset = CashedCustomDataset(dataset, dataset_dir, split="test-align", percent=percent,
                                    transform=get_transforms(dataset, blur=0, color=1, test=True, exclude_to_tensor=True))
test_conflict_dataset = CashedCustomDataset(dataset, dataset_dir, split="test-conflict", percent=percent,
                                    transform=get_transforms(dataset, blur=0, color=1, test=True, exclude_to_tensor=True))
test_dataset = CashedCustomDataset(dataset, dataset_dir, split="test", percent=percent,
                                transform=get_transforms(dataset, blur=0, color=1, test=True, exclude_to_tensor=True))

# print the number of samples in each dataset
print(f"Train Dataset Size: {len(train_dataset)}")
print(f"Train Degradation Dataset Size: {len(train_degradation_dataset)}")
print(f"Test Align Dataset Size: {len(test_align_dataset)}")
print(f"Test Conflict Dataset Size: {len(test_conflict_dataset)}")
print(f"Test Dataset Size: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
train_degradation_loader = DataLoader(train_degradation_dataset, batch_size=128, shuffle=True, num_workers=0)
test_align_loader = DataLoader(test_align_dataset, batch_size=128, shuffle=False, num_workers=0)
test_conflict_loader = DataLoader(test_conflict_dataset, batch_size=128, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

In [ ]:
dir = "cifar10c"
num_net = 10

figure_dir = os.path.join("..", "figures", dir)
save_dir = os.path.join("..", "results", dir)
if not os.path.exists(figure_dir):
    os.makedirs(figure_dir)
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

In [ ]:
criterion = nn.CrossEntropyLoss()

epochs = 100

alpha = 0.1  # LR suppression multiplier 
lr_suppress_start = epochs // 20  # 5%
lr_suppress_end = epochs // 10  # 10%

In [ ]:
model_wo = [resnet18().to(device) for _ in range(num_net)]
model_w = [resnet18().to(device) for _ in range(num_net)]
optimizer_wo = [torch.optim.Adam(model_wo[net_idx].parameters(), lr=0.0001) for net_idx in range(num_net)]
optimizer_w = [torch.optim.Adam(model_w[net_idx].parameters(), lr=0.0001) for net_idx in range(num_net)]
schedular_wo = [torch.optim.lr_scheduler.StepLR(optimizer_wo[net_idx], step_size=50, gamma=0.1) for net_idx in range(num_net)]
schedular_w = [torch.optim.lr_scheduler.StepLR(optimizer_w[net_idx], step_size=50, gamma=0.1) for net_idx in range(num_net)]

training_info = {"train_loss" : [], "train_acc" : [], "test_align_loss" : [], "test_align_acc" : [],
                 "test_conflict_loss" : [], "test_conflict_acc" : [], "test_loss" : [], "test_acc" : [],
                 "best_net" : None}

training_info_wo = [copy.deepcopy(training_info) for _ in range(num_net)]
training_info_w = [copy.deepcopy(training_info) for _ in range(num_net)]

In [ ]:
# random-noise warm-up (Random2, model_w에만 적용 — real-data epoch 카운트와 분리된 독립 pretraining)
from src.training.random_training import random_train

epochs_noise = 5
num_noise = 50000
input_shape = (3, 32, 32)   # CIFAR-10
output_size = 10

optimizer_noise = [torch.optim.SGD(model_w[net_idx].parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4) for net_idx in range(num_net)]

for net_idx in range(num_net):
    print(f"Random-noise warm-up Network {net_idx+1}/{num_net}")
    for epoch in range(epochs_noise):
        noise_loss, noise_acc = random_train(
            model_w[net_idx], optimizer_noise[net_idx], criterion,
            input_shape, num_noise, batch_size=128, output_size=output_size,
            device=device
        )
        print(f"  Warm-up Epoch {epoch+1}/{epochs_noise} - Loss: {noise_loss:.4f}, Acc: {noise_acc:.4f}")

In [ ]:
# training with degradation
for net_idx in range(num_net):
    print(f"Training Network {net_idx+1}/{num_net} without degradation")

    best_net_model_w = None
    best_acc_w = 0.0

    for epoch in range(epochs+1):
        if lr_suppress_start <= epoch < lr_suppress_end:
            for g in optimizer_w[net_idx].param_groups:
                g['lr'] = g['lr'] * alpha # lr suppression 
        if epoch == 0:
            train_loss, train_acc = test(model_w[net_idx], train_degradation_loader, criterion, device)
        elif epoch < epochs // 10:
            train_loss, train_acc = train(model_w[net_idx], train_degradation_loader, optimizer_w[net_idx], criterion, schedular_w[net_idx], device)
        else:
            train_loss, train_acc = train(model_w[net_idx], train_loader, optimizer_w[net_idx], criterion, schedular_w[net_idx], device)
        test_align_loss, test_align_acc = test(model_w[net_idx], test_align_loader, criterion, device)
        test_loss, test_acc = test(model_w[net_idx], test_loader, criterion, device)

        training_info_w[net_idx]["train_loss"].append(train_loss)
        training_info_w[net_idx]["train_acc"].append(train_acc)
        training_info_w[net_idx]["test_align_loss"].append(test_align_loss)
        training_info_w[net_idx]["test_align_acc"].append(test_align_acc)
        training_info_w[net_idx]["test_loss"].append(test_loss)
        training_info_w[net_idx]["test_acc"].append(test_acc)

        # save model parameters
        torch.save(model_w[net_idx].state_dict(), os.path.join(save_dir, f"model_w_epoch_{net_idx}_{epoch}.pth"))

        if test_acc > best_acc_w:
            best_acc_wo = test_acc
            training_info_w[net_idx]["best_net"] = copy.deepcopy(model_w[net_idx].state_dict())

        print(f"Epoch {epoch+1}/{epochs} - w: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

In [ ]:
# training without degradation
for net_idx in range(num_net):
    print(f"Training Network {net_idx+1}/{num_net} without degradation")

    best_net_model_wo = None
    best_acc_wo = 0.0

    for epoch in range(epochs+1):
        if epoch == 0:
            train_loss, train_acc = test(model_wo[net_idx], train_loader, criterion, device)
        else:
            train_loss, train_acc = train(model_wo[net_idx], train_loader, optimizer_wo[net_idx], criterion, schedular_wo[net_idx], device)
        test_align_loss, test_align_acc = test(model_wo[net_idx], test_align_loader, criterion, device)
        test_loss, test_acc = test(model_wo[net_idx], test_loader, criterion, device)

        training_info_wo[net_idx]["train_loss"].append(train_loss)
        training_info_wo[net_idx]["train_acc"].append(train_acc)
        training_info_wo[net_idx]["test_align_loss"].append(test_align_loss)
        training_info_wo[net_idx]["test_align_acc"].append(test_align_acc)
        training_info_wo[net_idx]["test_loss"].append(test_loss)
        training_info_wo[net_idx]["test_acc"].append(test_acc)

        torch.save(model_wo[net_idx].state_dict(), os.path.join(save_dir, f"model_wo_epoch_{net_idx}_{epoch}.pth"))

        if test_acc > best_acc_wo:
            best_acc_wo = test_acc
            training_info_wo[net_idx]["best_net"] = copy.deepcopy(model_wo[net_idx].state_dict())

        print(f"Epoch {epoch+1}/{epochs} - wo: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

In [ ]:
from custom.figure import plot_error

plt.figure(figsize=(60*mm, 30*mm))
data = [training_info_w[net_idx]["train_loss"] for net_idx in range(num_net)]
plot_error(data, color=color["sky"])
data = [training_info_wo[net_idx]["train_loss"] for net_idx in range(num_net)]
plot_error(data, color=color["orange"])
plt.axvline(x=epochs//10, linestyle='--', linewidth=0.5)
plt.ylim(0, 2.5)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.savefig(os.path.join(figure_dir, "train_loss.svg"))

plt.figure(figsize=(40*mm, 30*mm))
data = [training_info_w[net_idx]["train_acc"] for net_idx in range(num_net)]
plot_error(data, color=color["sky"])
data = [training_info_wo[net_idx]["train_acc"] for net_idx in range(num_net)]
plot_error(data, color=color["orange"])
plt.axvline(x=epochs//10, linestyle='--', linewidth=0.5)
plt.ylim(0, 1)
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training Accuracy")
plt.savefig(os.path.join(figure_dir, "train_acc.svg"))

plt.figure(figsize=(40*mm, 30*mm))
data = [training_info_w[net_idx]["test_align_acc"] for net_idx in range(num_net)]
plot_error(data, color=color["sky"])
data = [training_info_wo[net_idx]["test_align_acc"] for net_idx in range(num_net)]
plot_error(data, color=color["orange"])
plt.axvline(x=epochs//10, linestyle='--', linewidth=0.5)
plt.ylim(0, 1)
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Debiased test set accuracy")
plt.savefig(os.path.join(figure_dir, "biased_test_acc.svg"))

plt.figure(figsize=(40*mm, 30*mm))
data = [training_info_w[net_idx]["test_acc"] for net_idx in range(num_net)]
plot_error(data, color=color["sky"])
data = [training_info_wo[net_idx]["test_acc"] for net_idx in range(num_net)]
plot_error(data, color=color["orange"])
plt.axvline(x=epochs//10, linestyle='--', linewidth=0.5)
plt.ylim(0.1, 0.4)
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Debiased test set accuracy")
plt.savefig(os.path.join(figure_dir, "debiased_test_acc.svg"))

In [ ]:
max_w = [max(training_info_w[net_idx]["test_align_acc"]) for net_idx in range(num_net)]
max_wo = [max(training_info_wo[net_idx]["test_align_acc"]) for net_idx in range(num_net)]

plt.figure(figsize=(30*34/84*mm, 30*mm))
plt.bar([0, 1],
        [np.mean(max_wo), np.mean(max_w)],
        yerr=[np.std(max_wo), np.std(max_w)],
        tick_label=["w/o", "w/"],
        color=[color["orange"], color["sky"]],
        width=0.4)
plt.xticks(rotation=45, ha='right')
plt.xlim(-1, 2)
plt.ylim(0, 1.1)
plt.ylabel("Accuracy")
plt.title("Benchmark")
plt.savefig(os.path.join(figure_dir, "benchmark_.svg"))

# ranksum test
from scipy.stats import ranksums, ttest_ind
stat, p_value = ttest_ind(max_wo, max_w)
print(f"Ranksum test statistic: {stat}, p-value: {p_value}")

In [ ]:
max_w = [training_info_w[net_idx]["test_acc"][-1] for net_idx in range(num_net)]
max_wo = [training_info_wo[net_idx]["test_acc"][-1] for net_idx in range(num_net)]

hex = [13.87/100, 0.06/100]
end = [22.89/100, 0.27/100]
rebias = [22.27/100, 0.41/100]
lfF = [28.57/100, 1.30/100]
disent = [29.95/100, 0.71/100]

plt.figure(figsize=(30*34/84*mm, 30*mm))
plt.bar([3, 4, 5, 6, 7],
        [hex[0], end[0], rebias[0], lfF[0], disent[0]],
        yerr=[hex[1], end[1], rebias[1], lfF[1], disent[1]],
        tick_label=["HEX", "EnD", "ReBias", "LfF", "DisEnt"],
        color=[color["darkgray"], color["darkgray"], color["darkgray"], color["darkgray"], color["darkgray"]],
        width=0.8)
plt.axhline(y=np.mean(max_wo), color=color["orange"], linestyle='--', linewidth=0.5, label="w/o")
plt.axhline(y=np.mean(max_w), color=color["sky"], linestyle='--', linewidth=0.5, label="w/")
plt.xticks(rotation=45, ha='right')
plt.xlim(2, 8)
plt.ylim(0.1, 0.4)
plt.ylabel("Accuracy")
plt.title("Benchmark")
plt.savefig(os.path.join(figure_dir, "benchmark__.svg"))

print(f"Best Test Align Accuracy without degradation: {np.mean(max_wo)*100:.4f} (+/- {np.std(max_wo)*100:.4f})")
print(f"Best Test Align Accuracy with degradation: {np.mean(max_w)*100:.4f} (+/- {np.std(max_w)*100:.4f})")
# stat test
print(ttest_ind(max_wo, max_w))
print(ranksums(max_wo, max_w))

In [ ]:
align_w =  [training_info_w[net_idx]["test_align_acc"][-1] for net_idx in range(num_net)]
align_wo = [training_info_wo[net_idx]["test_align_acc"][-1] for net_idx in range(num_net)]

print(f"Best Test Align Accuracy without degradation: {np.mean(align_wo)*100:.4f} (+/- {np.std(align_wo)*100:.4f})")
print(f"Best Test Align Accuracy with degradation: {np.mean(align_w)*100:.4f} (+/- {np.std(align_w)*100:.4f})")
# stat test
print(ttest_ind(align_wo, align_w))
print(ranksums(align_wo, align_w))

In [ ]:
def evaluate_decision_rate(model, test_conflict_loader):
    model.eval()

    labels_list, attrs_list, preds_list = [], [], []
    with torch.no_grad():
        for inputs, labels, attrs in test_conflict_loader:
            inputs, labels, attrs = inputs.to(device), labels.to(device), attrs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            labels_list.append(labels.cpu().numpy())
            attrs_list.append(attrs.cpu().numpy())
            preds_list.append(preds.cpu().numpy())

    labels_list = np.concatenate(labels_list)
    attrs_list = np.concatenate(attrs_list)
    preds_list = np.concatenate(preds_list)

    decision_rates = []

    # Calculate decision rates for each class
    for i in range(10):
        # Get the indices of the samples belonging to class i
        class_indices = np.where(labels_list == i)[0]
        temp_preds = preds_list[class_indices]
        temp_attrs = attrs_list[class_indices]
        temp_labels = labels_list[class_indices]

        # Calculate the decision rate to the intrinsic features vs spurious bias
        correct_labels = np.sum(temp_preds == temp_labels)
        correct_attrs = np.sum(temp_preds == temp_attrs)
        decision_rate = correct_labels / (correct_labels + correct_attrs)
        decision_rates.append(decision_rate)
    return decision_rates

decision_rates_wo = []
decision_rates_w = []

for net_idx in range(num_net):
    print(f"Evaluating Decision Rate for Network {net_idx+1}/{num_net}")

    model_wo = resnet18().to(device)
    best_idx_wo = np.argmax(training_info_wo[net_idx]["test_align_acc"])
    model_wo.load_state_dict(torch.load(os.path.join(save_dir, f"model_wo_epoch_{net_idx}_{best_idx_wo}.pth")))
    
    model_w = resnet18().to(device)
    best_idx_w = np.argmax(training_info_w[net_idx]["test_align_acc"])
    model_w.load_state_dict(torch.load(os.path.join(save_dir, f"model_w_epoch_{net_idx}_{best_idx_w}.pth")))

    decision_rates_wo.append(evaluate_decision_rate(model_wo, test_conflict_loader))
    decision_rates_w.append(evaluate_decision_rate(model_w, test_conflict_loader))

decision_rates_wo = np.array(decision_rates_wo)
decision_rates_w = np.array(decision_rates_w)

In [ ]:
plt.figure(figsize=(30*mm, 50*mm))
plt.errorbar(x=decision_rates_wo.mean(axis=0), y=range(10),
            xerr=decision_rates_wo.std(axis=0),
            color=color["orange"], fmt='o', capsize=0, label="w/o", ecolor='k', elinewidth=0.5, markersize=2.5)
plt.errorbar(x=decision_rates_w.mean(axis=0), y=range(10),
            xerr=decision_rates_w.std(axis=0),
            color=color["sky"], fmt='o', capsize=0, label="w/", ecolor='k', elinewidth=0.5, markersize=2.5)
for i in range(decision_rates_wo.shape[0]):
    plt.scatter(decision_rates_wo[i, :], np.arange(0, 10) + np.random.normal(0, 0.1, 10), color=color["orange"], s=5, alpha=0.3, edgecolors='None', marker='o')
    plt.scatter(decision_rates_w[i, :], np.arange(0, 10) + np.random.normal(0, 0.1, 10), color=color["sky"], s=5, alpha=0.3, edgecolors='None', marker='o')

plt.xlim(0, 1)
plt.ylim(-0.5, 9.5)
plt.yticks(range(10))
plt.gca().invert_yaxis()
plt.gca().invert_xaxis()
plt.xlabel("Decision Rate")
plt.ylabel("Class")
plt.savefig(os.path.join(figure_dir, "decision_rate.svg"))

In [ ]:
import json
with open(os.path.join("..", "human_experiment", "decision_rate_cifar10_summary.json"), 'r') as f:
    human_decision_rate = json.load(f)
human_decision_rate = np.array(human_decision_rate["decision_rate_summary"])

plt.figure(figsize=(30*mm, 50*mm))
plt.errorbar(x=decision_rates_wo.mean(axis=0), y=range(10),
            xerr=decision_rates_wo.std(axis=0),
            color=color["orange"], fmt='o', capsize=0, label="w/o", ecolor='k', elinewidth=0.5, markersize=2.5)
plt.errorbar(x=decision_rates_w.mean(axis=0), y=range(10),
            xerr=decision_rates_w.std(axis=0),
            color=color["sky"], fmt='o', capsize=0, label="w/", ecolor='k', elinewidth=0.5, markersize=2.5)
plt.errorbar(x=human_decision_rate.mean(axis=0), y=range(10),
            xerr=human_decision_rate.std(axis=0),
            color="#3C319B", fmt='D', capsize=0, label="Human", ecolor='k', elinewidth=0.5, markersize=2.5)
for i in range(human_decision_rate.shape[0]):
    plt.scatter(human_decision_rate[i, :], np.arange(0, 10) + np.random.normal(0, 0.1, 10), color="#3C319B", s=5, alpha=0.3, edgecolors='None', marker='D')
    plt.scatter(decision_rates_wo[i, :], np.arange(0, 10) + np.random.normal(0, 0.1, 10), color=color["orange"], s=5, alpha=0.3, edgecolors='None', marker='o')
    plt.scatter(decision_rates_w[i, :], np.arange(0, 10) + np.random.normal(0, 0.1, 10), color=color["sky"], s=5, alpha=0.3, edgecolors='None', marker='o')

plt.xlim(0, 1)
plt.ylim(-0.5, 9.5)
plt.yticks(range(10))
plt.gca().invert_yaxis()
plt.gca().invert_xaxis()
plt.xlabel("Decision Rate")
plt.ylabel("Class")
plt.savefig(os.path.join(figure_dir, "decision_rate.svg"))

In [ ]:
plt.figure(figsize=(113/85*30*mm, 218/85*30*mm))
plt.errorbar(x=decision_rates_wo.mean(axis=0), y=range(10),
            xerr=decision_rates_wo.std(axis=0),
            color=color["orange"], fmt='o', capsize=0, label="w/o", ecolor='k', elinewidth=0.5, markersize=3)
plt.errorbar(x=human_decision_rate.mean(axis=0), y=range(10),
            xerr=human_decision_rate.std(axis=0),
            color="#3C319B", fmt='D', capsize=0, label="Human", ecolor='k', elinewidth=0.5, markersize=3)

for i in range(human_decision_rate.shape[0]):
    plt.scatter(human_decision_rate[i, :], np.arange(0, 10) + np.random.normal(0, 0.1, 10), color="#3C319B", s=5, alpha=0.5, edgecolors='None', marker='D')
    plt.scatter(decision_rates_wo[i, :], np.arange(0, 10) + np.random.normal(0, 0.1, 10), color=color["orange"], s=5, alpha=0.5, edgecolors='None', marker='o')

plt.xlim(0, 1)
plt.ylim(-0.5, 9.5)
plt.yticks(range(10))
plt.gca().invert_yaxis()
plt.gca().invert_xaxis()
plt.xlabel("Decision Rate")
plt.ylabel("Class")
plt.savefig(os.path.join(figure_dir, "decision_rate_.svg"))

In [ ]:
plt.figure(figsize=(113/85*30*mm, 30*mm))

for i in range(human_decision_rate.shape[0]):
    plt.scatter(human_decision_rate[i, :], np.arange(0, 10) + np.random.normal(0, 0.1, 10), color="#3C319B", s=3, alpha=0.5, edgecolors='None', marker='D')
    plt.scatter(decision_rates_wo[i, :], np.arange(0, 10) + np.random.normal(0, 0.1, 10), color=color["orange"], s=3, alpha=0.5, edgecolors='None', marker='o')
    print(ranksums(human_decision_rate[i, :], decision_rates_wo[i, :]))

plt.boxplot(human_decision_rate.flatten(), vert=False, positions=[11], widths=1, boxprops=dict(facecolor="#3C319B", linewidth=0))
plt.boxplot(decision_rates_wo.flatten(), vert=False, positions=[11], widths=1, boxprops=dict(facecolor=color["orange"], linewidth=0))
print(ranksums(human_decision_rate.flatten(), decision_rates_wo.flatten()))

plt.xlim(0, 1)
plt.ylim(-0.5, 12)
plt.yticks([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, "Summary"])
plt.gca().invert_yaxis()
plt.gca().invert_xaxis()
plt.xlabel("Decision Rate")
plt.ylabel("Class")
plt.savefig(os.path.join(figure_dir, "decision_rate__.svg"))

In [ ]:
# measure feature vector of penultimate layer (using test set)

def measure_feature_vector(model, test_loader, target_layers):
    # Create a deep copy of the network to safely register hooks.
    model_copy = copy.deepcopy(model).to(device)

    # Initialize dictionary to store features for each target layer
    features_dict = {layer: [] for layer in target_layers}
    hooks = []

    # Register hooks to capture output from each target layer
    for name, module in model_copy.named_modules():
        if name in target_layers:
            def hook_fn(module, input, output, key=name):
                features_dict[key].append(torch.flatten(output, 1).detach().cpu().numpy())
            hooks.append(module.register_forward_hook(hook_fn))

    labels_list, attrs_list = [], []

    model_copy.eval()
    with torch.no_grad():
        for inputs, labels, attrs in test_loader:
            inputs, labels, attrs = inputs.to(device), labels.to(device), attrs.to(device)
            _ = model_copy(inputs)  # Forward pass (hooks capture outputs)
            labels_list.append(labels.cpu().numpy())
            attrs_list.append(attrs.cpu().numpy())

    # Remove hooks to avoid side effects
    for h in hooks:
        h.remove()

    # Concatenate features across batches for each target layer
    for key in features_dict:
        features_dict[key] = np.concatenate(features_dict[key], axis=0)

    labels_array = np.concatenate(labels_list, axis=0)
    attrs_array = np.concatenate(attrs_list, axis=0)

    return features_dict, labels_array, attrs_array

In [ ]:
modules = []
for name, module in model_wo.named_modules():
    modules.append(name)

print(modules)

In [ ]:
small_test_dataset = copy.deepcopy(test_dataset)

num_samples = 1000
rand_idx = np.random.choice(len(small_test_dataset), num_samples, replace=False)
temp_data = [small_test_dataset.data[i] for i in rand_idx]
small_test_dataset.data = temp_data
small_test_loader = DataLoader(small_test_dataset, batch_size=128, shuffle=False, num_workers=0)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

def separability(X, y, cv=5, C=1.0, random_state=0):
    """
    Measure separability of the points X (n_samples×n_features) 
    w.r.t. labels y using a linear SVM.

    Returns
    -------
    acc_mean : float
        mean cross-validated accuracy over `cv` folds
    """
    clf = make_pipeline(
        StandardScaler(),
        LinearSVC(C=C, max_iter=10_000, random_state=random_state)
    )
    scores = cross_val_score(clf, X, y, cv=cv)
    acc_mean = scores.mean()

    return acc_mean

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sc_wo_intrinsic_ = []
sc_w_intrinsic_ = []
sc_wo_bias_ = []
sc_w_bias_ = []

for net_idx in range(num_net):
    print(f"Calculating Silhouette Score for Network {net_idx+1}/{num_net}")

    silhouette_score_wo_intrinsic = []
    silhouette_score_w_intrinsic = []
    silhouette_score_wo_bias = []
    silhouette_score_w_bias = []

    for epoch in range(epochs):
        temp_model_wo = resnet18().to(device)
        temp_model_wo.load_state_dict(torch.load(os.path.join(save_dir, f"model_wo_epoch_{net_idx}_{epoch}.pth")))
        temp_model_w = resnet18().to(device)
        temp_model_w.load_state_dict(torch.load(os.path.join(save_dir, f"model_w_epoch_{net_idx}_{epoch}.pth")))

        target_layers = ["fc"]
        features_wo, labels_wo, attrs_wo = measure_feature_vector(temp_model_wo, small_test_loader, target_layers)
        features_w, labels_w, attrs_w = measure_feature_vector(temp_model_w, small_test_loader, target_layers)

        pca_wo = PCA(n_components=10).fit_transform(features_wo["fc"])
        pca_w = PCA(n_components=10).fit_transform(features_w["fc"])

        temp_silhouette_score_wo_intrinsic = separability(pca_wo, labels_wo)
        temp_silhouette_score_w_intrinsic = separability(pca_w, labels_w)
        temp_silhouette_score_wo_bias = separability(pca_wo, attrs_wo)
        temp_silhouette_score_w_bias = separability(pca_w, attrs_w)

        silhouette_score_wo_intrinsic.append(temp_silhouette_score_wo_intrinsic)
        silhouette_score_w_intrinsic.append(temp_silhouette_score_w_intrinsic)
        silhouette_score_wo_bias.append(temp_silhouette_score_wo_bias)
        silhouette_score_w_bias.append(temp_silhouette_score_w_bias)

        print(f"Epoch {epoch} - w/o intrinsic: {temp_silhouette_score_wo_intrinsic:.4f}, w intrinsic: {temp_silhouette_score_w_intrinsic:.4f}, w/o bias: {temp_silhouette_score_wo_bias:.4f}, w bias: {temp_silhouette_score_w_bias:.4f}")
        
    sc_wo_intrinsic_.append(silhouette_score_wo_intrinsic)
    sc_w_intrinsic_.append(silhouette_score_w_intrinsic)
    sc_wo_bias_.append(silhouette_score_wo_bias)
    sc_w_bias_.append(silhouette_score_w_bias)

sc_wo_intrinsic_ = np.array(sc_wo_intrinsic_)
sc_w_intrinsic_ = np.array(sc_w_intrinsic_)
sc_wo_bias_ = np.array(sc_wo_bias_)
sc_w_bias_ = np.array(sc_w_bias_)

In [ ]:
# intrinsic manifold separability
plt.figure(figsize=(40*mm, 30*mm))
plot_error(sc_w_intrinsic_, color=color["sky"], label="w/")
plot_error(sc_wo_intrinsic_, color=color["orange"], label="w/o")
plt.axvline(x=epochs//10, linestyle='--', linewidth=0.5)
plt.axhline(y=0, linestyle='--', linewidth=0.5)
plt.xlabel("Epochs")
plt.ylabel("SVM performance")
plt.title("Intrinsic Manifold Separability")
plt.savefig(os.path.join(figure_dir, "intrinsic_manifold_separability_svm.svg"))

print(ranksums(sc_wo_intrinsic_[:, -1], sc_w_intrinsic_[:, -1]))

# bias manifold separability
plt.figure(figsize=(40*mm, 30*mm))
plot_error(sc_w_bias_, color=color["sky"], label="w/")
plot_error(sc_wo_bias_, color=color["orange"], label="w/o")
plt.axvline(x=epochs//10, linestyle='--', linewidth=0.5)
plt.axhline(y=0, linestyle='--', linewidth=0.5)
plt.ylabel("SVM performance")
plt.title("Bias Manifold Separability")
plt.savefig(os.path.join(figure_dir, "bias_manifold_separability_svm.svg"))

print(ranksums(sc_wo_bias_[:, -1], sc_w_bias_[:, -1]))

In [ ]:
net_idx = 0

model_wo = resnet18().to(device)
model_wo.load_state_dict(torch.load(os.path.join(save_dir, f"model_wo_epoch_{net_idx}_80.pth")))
model_w = resnet18().to(device)
model_w.load_state_dict(torch.load(os.path.join(save_dir, f"model_w_epoch_{net_idx}_80.pth")))

features_wo, labels_wo, attrs_wo = measure_feature_vector(model_wo, test_loader, ["fc"])
features_w, labels_w, attrs_w = measure_feature_vector(model_w, test_loader, ["fc"])

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

tsne_wo = TSNE(n_components=2, random_state=1).fit_transform(features_wo["fc"])
tsne_w = TSNE(n_components=2, random_state=1).fit_transform(features_w["fc"])

In [ ]:
# Plotting the t-SNE results
random_order = np.random.permutation(len(tsne_wo))
plt.figure(figsize=(30*mm, 30*mm))
plt.scatter(tsne_wo[random_order, 0], tsne_wo[random_order, 1], c=labels_wo[random_order], cmap='tab10', s=0.1, label="w/o")
plt.title("t-SNE (w/o, intrinsic features)")
plt.xlabel("t-SNE axis 1")
plt.ylabel("t-SNE axis 2")
plt.xlim(-100, 100)
plt.ylim(-100, 100)
plt.savefig(os.path.join(figure_dir, "tsne_wo_intrinsic.png"), dpi=300)

plt.figure(figsize=(30*mm, 30*mm))
plt.scatter(tsne_w[random_order, 0], tsne_w[random_order, 1], c=labels_w[random_order], cmap='tab10', s=0.1, label="w/")
plt.title("t-SNE (w/, intrinsic features)")
plt.xlabel("t-SNE axis 1")
plt.ylabel("t-SNE axis 2")
plt.xlim(-100, 100)
plt.ylim(-100, 100)
plt.savefig(os.path.join(figure_dir, "tsne_w_intrinsic.png"), dpi=300)

In [ ]:
bias_color = ["#EA3323", "#EF8633", "#FFFE54", "#A1FB4E", "#75FA4C", "#74FBFD", "#001AF5", "#7424F5", "#EA347F", "#EA3AF7" ]
# custom cmap
from matplotlib.colors import ListedColormap
bias_cmap = ListedColormap(bias_color)

# Plotting the t-SNE results
plt.figure(figsize=(30*mm, 30*mm))
plt.scatter(tsne_wo[random_order, 0], tsne_wo[random_order, 1], c=attrs_wo[random_order], cmap=bias_cmap, s=0.1, label="w/o")
plt.title("t-SNE (w/o, spurious bias)")
plt.xlabel("t-SNE axis 1")
plt.ylabel("t-SNE axis 2")
plt.xlim(-100, 100)
plt.ylim(-100, 100)
plt.savefig(os.path.join(figure_dir, "tsne_wo_spurious.png"), dpi=300)

plt.figure(figsize=(30*mm, 30*mm))
plt.scatter(tsne_w[random_order, 0], tsne_w[random_order, 1], c=attrs_w[random_order], cmap=bias_cmap, s=0.1, label="w/")
plt.title("t-SNE (w/, spurious bias)")
plt.xlabel("t-SNE axis 1")
plt.ylabel("t-SNE axis 2")
plt.xlim(-100, 100)
plt.ylim(-100, 100)
plt.savefig(os.path.join(figure_dir, "tsne_w_spurious.png"), dpi=300)

In [ ]:
from sklearn import svm
from sklearn.metrics import accuracy_score
from sklearn.metrics import silhouette_samples

def evaluate_class_separability(representation, labels):
    silhouette_sample_results = silhouette_samples(representation, labels)

    silhouette_score_list = []

    for i in range(10):
        # Get the indices of the samples belonging to class i
        class_indices = np.where(labels == i)[0]
        temp_silhouette_samples = silhouette_sample_results[class_indices]

        # Calculate the average silhouette score for class i
        avg_silhouette_score = np.mean(temp_silhouette_samples)
        silhouette_score_list.append(avg_silhouette_score)

    return silhouette_score_list


def evaluate_representation_rate(linear_sep_labels, linear_sep_attrs):
    representation_rate = []
    for i in range(10):
        a = linear_sep_labels[i]
        b = linear_sep_attrs[i]
        k = 5
        rate =  1 / (1 + np.exp(-k * (a - b)))
        representation_rate.append(rate)

    return representation_rate

In [ ]:
import warnings
warnings.filterwarnings("ignore")

representation_rates_wo = []
representation_rates_w = []

for net_idx in range(num_net):
    model_wo = resnet18().to(device)
    model_wo.load_state_dict(torch.load(os.path.join(save_dir, f"model_wo_epoch_{net_idx}_80.pth")))
    model_w = resnet18().to(device)
    model_w.load_state_dict(torch.load(os.path.join(save_dir, f"model_w_epoch_{net_idx}_80.pth")))

    features_wo, labels_wo, attrs_wo = measure_feature_vector(model_wo, test_loader, ["fc"])
    features_w, labels_w, attrs_w = measure_feature_vector(model_w, test_loader, ["fc"])

    linear_sep_labels_wo = evaluate_class_separability(features_wo["fc"], labels_wo)
    linear_sep_attrs_wo = evaluate_class_separability(features_wo["fc"], attrs_wo)

    linear_sep_labels_w = evaluate_class_separability(features_w["fc"], labels_w)
    linear_sep_attrs_w = evaluate_class_separability(features_w["fc"], attrs_w)

    representation_rates_wo.append(evaluate_representation_rate(linear_sep_labels_wo, linear_sep_attrs_wo))
    representation_rates_w.append(evaluate_representation_rate(linear_sep_labels_w, linear_sep_attrs_w))

representation_rates_wo = np.array(representation_rates_wo)
representation_rates_w = np.array(representation_rates_w)

In [ ]:
plt.figure(figsize=(30*mm, 50*mm))
plt.errorbar(x=representation_rates_wo.mean(axis=0), y=range(10),
            xerr=representation_rates_wo.std(axis=0),
            color=color["orange"], fmt='o', capsize=0, label="w/o", ecolor='k', elinewidth=0.5, markersize=2.5)
plt.errorbar(x=representation_rates_w.mean(axis=0), y=range(10),
            xerr=representation_rates_w.std(axis=0),
            color=color["sky"], fmt='o', capsize=0, label="w/", ecolor='k', elinewidth=0.5, markersize=2.5)
for i in range(representation_rates_wo.shape[0]):
    plt.scatter(representation_rates_wo[i, :], np.arange(0, 10) + np.random.normal(0, 0.1, 10), color=color["orange"], s=5, alpha=0.3, edgecolors='None', marker='o')
    plt.scatter(representation_rates_w[i, :], np.arange(0, 10) + np.random.normal(0, 0.1, 10), color=color["sky"], s=5, alpha=0.3, edgecolors='None', marker='o')


plt.xlim(0, 1)
plt.ylim(-0.5, 9.5)
plt.yticks(range(10))
plt.gca().invert_yaxis()
plt.gca().invert_xaxis()
plt.xlabel("Representation Rate")
plt.ylabel("Class")
plt.savefig(os.path.join(figure_dir, "representation_rate_wo.svg"))

In [ ]:
from sklearn.linear_model import LinearRegression

plt.figure(figsize=(30*mm, 30*mm))
plt.errorbar(representation_rates_wo.mean(axis=0), decision_rates_wo.mean(axis=0),
            xerr=representation_rates_wo.std(axis=0),
            yerr=decision_rates_wo.std(axis=0),
            color=color["orange"], fmt='o', capsize=0, label="w/", ecolor='k', elinewidth=0.5, markersize=2)
plt.errorbar(representation_rates_w.mean(axis=0), decision_rates_w.mean(axis=0),
            xerr=representation_rates_w.std(axis=0),
            yerr=decision_rates_w.std(axis=0),
            color=color["sky"], fmt='o', capsize=0, label="w/o", ecolor='k', elinewidth=0.5, markersize=2)

# concatenate
X = np.concatenate([representation_rates_w.mean(axis=0), representation_rates_wo.mean(axis=0)])
Y = np.concatenate([decision_rates_w.mean(axis=0), decision_rates_wo.mean(axis=0)])

model = LinearRegression().fit(X.reshape(-1, 1), Y.reshape(-1, 1))
xrange = np.linspace(0, 1, 100)
plt.plot(xrange, model.predict(xrange.reshape(-1, 1)), color='k', linestyle='--', linewidth=0.5)

print("Coefficient:", model.coef_[0][0])
print("Intercept:", model.intercept_[0])
print("R^2 score:", model.score(X.reshape(-1, 1), Y.reshape(-1, 1)))


from scipy.stats import pearsonr
corr, p_value = pearsonr(X.reshape(-1, 1), Y.reshape(-1, 1))
print("Pearson correlation (w/):", corr, "p-value:", p_value)

plt.xlim(0, 0.7)
plt.ylim(0, 0.7)
plt.xlabel("Representation Rate")
plt.ylabel("Decision Rate")
plt.savefig(os.path.join(figure_dir, "representation_rate_vs_decision_rate.svg"))

